# Chapter 18 &mdash; Fixpoint Equations and the $Y$ Combinator

**Concept 7 of the Chapter 18 decomposition:** *Fixpoint Equations and the $Y$ Combinator*

$Y = \lambda f.(\lambda x.\,f(x\,x))(\lambda x.\,f(x\,x))$ satisfies $Y\,G = G\,(Y\,G)$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18-Lambda/Concept-Y-Combinator/Concept-Y-Combinator.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


$$Y = \lambda f.\,(\lambda x.\,f\,(x\,x))\,(\lambda x.\,f\,(x\,x))$$

The defining property, by one $\beta$-step:

$$Y\,G \;\to\; (\lambda x.\,G\,(x\,x))(\lambda x.\,G\,(x\,x)) \;\to\; G\,\big((\lambda x.\,G\,(x\,x))(\lambda x.\,G\,(x\,x))\big) \;=\; G\,(Y\,G)$$

So $Y\,G$ **is** a fixpoint of $G$: exactly the $f$ with $f = G\,f$ that Concept 6
needed. Recursion, with **no names anywhere**.

The engine is **self-application** $x\,x$ &mdash; a term applying itself, which is what
lets a nameless function reach itself.

In a language with **eager** evaluation (Python, ML, Scheme), $Y$ as written diverges:
$x\,x$ is evaluated before it is needed. Concept 8 fixes that.

## 2. Definitions

### Y and its eager cousin

In [ ]:
# --- fixpoint combinators ------------------------------------------------
# Y diverges under Python's EAGER evaluation, because (x x) is evaluated
# before it is needed.  Y_e ("eager Y", also called Z) wraps the
# self-application in a lambda, delaying it until it is applied.
Y  = lambda f: (lambda x: f(x(x)))(lambda x: f(x(x)))          # loops in Python
Ye = lambda f: (lambda x: f(lambda v: x(x)(v)))(lambda x: f(lambda v: x(x)(v)))


G_fact = lambda f: lambda n: 1 if n == 0 else n * f(n - 1)

### Verifying the fixpoint property

In [ ]:
def is_fixpoint(G, f, inputs):
    return all(G(f)(n) == f(n) for n in inputs)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch18&nbsp;6.&nbsp;The Problem of Recursion in an Anonymous World](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18-Lambda/Concept-Recursion-Without-Names/Concept-Recursion-Without-Names.ipynb) &nbsp;&middot;&nbsp; [**Chapter 18** index](https://github.com/ganeshutah/Jove/blob/master/Chapter18-Lambda/README.md) &nbsp;&middot;&nbsp; [Ch18&nbsp;8.&nbsp;Eager versus Lazy Evaluation, and the Combinator $Y_e$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18-Lambda/Concept-Eager-Versus-Lazy/Concept-Eager-Versus-Lazy.ipynb)&nbsp;&rarr;

---

## 3. Tests

**$Y$ diverges in Python** &mdash; the very first thing to see.

In [ ]:
import sys
sys.setrecursionlimit(200)
try:
    Y(G_fact)
    print("no error (unexpected)")
except RecursionError:
    print("RecursionError -- Y(G) never returns under eager evaluation")
sys.setrecursionlimit(3000)

**$Y_e$** works: same idea, self-application delayed.

In [ ]:
fact = Ye(G_fact)
print([fact(n) for n in range(8)])
assert [fact(n) for n in range(8)] == [1, 1, 2, 6, 24, 120, 720, 5040]

And it really is a **fixpoint**: $G\,f$ and $f$ agree everywhere.

In [ ]:
print("is Ye(G) a fixpoint of G on 0..7 ?", is_fixpoint(G_fact, fact, range(8)))
assert is_fixpoint(G_fact, fact, range(8))
print("  G(fact)(5) =", G_fact(fact)(5), "   fact(5) =", fact(5))

**No names anywhere.** The whole thing is one expression.

In [ ]:
anonymous_factorial = (lambda f: (lambda x: f(lambda v: x(x)(v)))
                                 (lambda x: f(lambda v: x(x)(v))))(
                       lambda g: lambda n: 1 if n == 0 else n * g(n - 1))
print("5! =", anonymous_factorial(5))
print("8! =", anonymous_factorial(8))
assert anonymous_factorial(8) == 40320
print("\nNot one name is defined or referred to.  That is the point of Y.")

**Self-application** is the engine.

In [ ]:
selfapp = lambda x: x(x)
print("  (lambda x: x(x)) applied to the identity :", selfapp(lambda y: y)(7))
print()
print("x x is how a nameless term reaches itself.  It is also how you write")
print("Omega and diverge -- the same trick, used two ways.")

$Y$ works for **any** $G$, which is what makes it a combinator.

In [ ]:
G_fib  = lambda f: lambda n: n if n < 2 else f(n - 1) + f(n - 2)
G_sum  = lambda f: lambda n: 0 if n == 0 else n + f(n - 1)
G_len  = lambda f: lambda xs: 0 if not xs else 1 + f(xs[1:])
print("  fib  :", [Ye(G_fib)(n) for n in range(10)])
print("  sum  :", [Ye(G_sum)(n) for n in range(7)])
print("  len  :", Ye(G_len)([1, 2, 3, 4, 5]))
assert Ye(G_fib)(9) == 34 and Ye(G_sum)(5) == 15 and Ye(G_len)([1]*5) == 5

## 4. Exercises


1. Perform the $\beta$-reduction $Y\,G \to G\,(Y\,G)$ on paper.
2. Is $Y$ typeable in a simply-typed lambda calculus? Why not?
3. Find Turing's fixpoint combinator $\Theta$ and check it has the same property.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter18-Lambda/Concept-Y-Combinator')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')